# Chapter 6: Training optimization and DPO

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/zerokaraLLM/blob/main/notebooks/ch06_training_optimization_dpo.ipynb)

This notebook is generated from the complete chapter source. Nothing is replaced by a toy implementation. Python files are only split into notebook cells for readability; concatenating those cells reproduces the original source exactly.

**Pinned upstream commit:** `c9b6e2ed531b08dd9f451a091a34e9645148e2e2`


## Notebook architecture

The notebook follows the chapter as a readable pipeline rather than hiding implementation behind `%run` calls. Shared local modules used by the chapter are written from visible cells first, then every chapter script is presented in source order. Original model dimensions, algorithms, and training hyperparameters are preserved.


In [ ]:
from pathlib import Path
import os
import subprocess

UPSTREAM_COMMIT = 'c9b6e2ed531b08dd9f451a091a34e9645148e2e2'
WORKDIR = Path('/content/deep-learning-from-scratch-6')

if not WORKDIR.exists():
    subprocess.run(['git', 'clone', '--quiet', 'https://github.com/oreilly-japan/deep-learning-from-scratch-6.git', str(WORKDIR)], check=True)
    subprocess.run(['git', '-C', str(WORKDIR), 'checkout', '--quiet', UPSTREAM_COMMIT], check=True)

os.chdir(WORKDIR)
print('working directory:', Path.cwd())
try:
    import torch
    print('torch:', torch.__version__)
    print('cuda:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('torch check:', exc)


## Shared modules used by this chapter

These cells keep shared architecture visible while preserving the original package layout for imports.


### `storybot/model.py`

SHA-256: `6c6a0acca85129e8082f85ad041bbbf3cc150e37e81870e58328bbc1e0b0f0b7`


In [ ]:
%%writefile storybot/model.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class RoPE(nn.Module):
    def __init__(self, theta, key_dim, max_context_len):
        super().__init__()
        assert key_dim % 2 == 0
        half = key_dim // 2

        half_ids = torch.arange(0, half)
        inv_freq = 1.0 / (theta ** ( (2.0 * half_ids) / key_dim ))  # (half,)

        positions = torch.arange(max_context_len)  # (max_context_len,)
        angles = positions[:, None] * inv_freq[None, :]  # (max_context_len, half)

        cos = torch.cos(angles)  # (max_context_len, half)
        sin = torch.sin(angles)  # (max_context_len, half)

        self.register_buffer("cos_cache", cos)
        self.register_buffer("sin_cache", sin)

    def forward(self, x, offset=0):
        batch_size, num_head, context_len, key_dim = x.shape

        # 入力の型を保存し、float32で計算
        input_dtype = x.dtype
        x = x.float()

        # offsetを考慮して位置エンコーディングを取得
        max_context_len = self.cos_cache.size(0)
        if offset + context_len > max_context_len:
            offset = max_context_len - context_len

        cos = self.cos_cache[offset:offset + context_len]
        sin = self.sin_cache[offset:offset + context_len]

        x_even = x[..., 0::2]
        x_odd  = x[..., 1::2]

        x_rot_even = x_even * cos - x_odd * sin
        x_rot_odd  = x_even * sin + x_odd * cos

        out = torch.stack([x_rot_even, x_rot_odd], dim=-1)
        out = out.reshape(batch_size, num_head, context_len, key_dim)

        return out.to(input_dtype)  # 元の型に戻す

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, n_head, head_dim, rope=None):
        super().__init__()
        self.n_head = n_head
        self.head_dim = head_dim
        E, H, D = embed_dim, n_head, head_dim

        self.W_q = nn.Linear(E, H*D, bias=False)
        self.W_k = nn.Linear(E, H*D, bias=False)
        self.W_v = nn.Linear(E, H*D, bias=False)
        self.W_o = nn.Linear(H*D, E, bias=False)

        self.rope = rope

        # KV-Cache用の変数を追加
        self.k_cache = None  # Keyのキャッシュ
        self.v_cache = None  # Valueのキャッシュ
        self.cache_offset = 0  # 現在のキャッシュ位置を追跡

    def forward(self, x, use_cache=False):
        B, C, E = x.shape
        H, D = self.n_head, self.head_dim

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        Q = Q.view(B, C, H, D).transpose(1, 2)
        K = K.view(B, C, H, D).transpose(1, 2)
        V = V.view(B, C, H, D).transpose(1, 2)

        # RoPEにoffsetを渡す
        if self.rope is not None:
            if use_cache:
                Q = self.rope(Q, self.cache_offset)
                K = self.rope(K, self.cache_offset)
            else:
                Q = self.rope(Q)
                K = self.rope(K)

        # KV-Cacheの処理
        if use_cache:
            # Prefill（初回）かDecode（2回目以降）かを判定
            is_first_call = (self.k_cache is None)

            if is_first_call:
                # 初回:キャッシュを初期化
                self.k_cache = K
                self.v_cache = V
            else:
                # 2回目以降:新しいKeyとValueをキャッシュに追加
                self.k_cache = torch.cat([self.k_cache, K], dim=2)
                self.v_cache = torch.cat([self.v_cache, V], dim=2)

            # オフセットを更新(次のトークンの位置へ)
            self.cache_offset += C

            # キャッシュされた全てのKeyとValueを使用
            K = self.k_cache
            V = self.v_cache

        # 通常のAttention計算
        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / (D ** 0.5)

        # Causal Maskの適用
        # - 学習時（use_cache=False）: 常に適用
        # - Prefill（初回・プロンプト全体処理）: 適用（各トークンは前方のみ）
        # - Decode（2回目以降・1トークン生成）: 不要（新トークンは全キャッシュにattend）
        if not use_cache or (use_cache and is_first_call):
            mask = torch.tril(torch.ones(C, C, device=scores.device))
            scores = scores.masked_fill(mask == 0, float('-inf'))

        weights = F.softmax(scores, dim=-1)
        hidden = torch.matmul(weights, V)

        hidden = hidden.transpose(1, 2).contiguous()
        hidden = hidden.view(B, C, H * D)
        output = self.W_o(hidden)
        return output

    def clear_cache(self):
        """キャッシュをクリアする"""
        self.k_cache = None
        self.v_cache = None
        self.cache_offset = 0

def silu(x):
    return x * torch.sigmoid(x)

class SwiGLU(nn.Module):
    def __init__(self, x_dim, hidden_dim=None):
        super().__init__()
        if hidden_dim is None:
            hidden_dim = int(x_dim * 8 / 3)

        self.W = nn.Linear(x_dim, hidden_dim, bias=False)
        self.V = nn.Linear(x_dim, hidden_dim, bias=False)
        self.O = nn.Linear(hidden_dim, x_dim, bias=False)

    def forward(self, x):
        a = self.W(x)
        b = self.V(x)

        gated = F.silu(a) * b  # silu(a) * b
        out = self.O(gated)
        return out

class Block(nn.Module):
    def __init__(self, embed_dim, n_head, ff_dim, rope=None):
        super().__init__()
        head_dim = embed_dim // n_head
        self.norm1 = nn.RMSNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, n_head, head_dim, rope)
        self.norm2 = nn.RMSNorm(embed_dim)
        self.ffn = SwiGLU(embed_dim, ff_dim)

    def forward(self, x, use_cache=False):
        x = x + self.attn(self.norm1(x), use_cache=use_cache)
        x = x + self.ffn(self.norm2(x))
        return x

    def clear_cache(self):
        """キャッシュをクリアする"""
        self.attn.clear_cache()


class GPT(nn.Module):
    def __init__(self, vocab_size, max_context_len, embed_dim, n_head, n_layer, ff_dim, theta=10000):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_context_len = max_context_len
        self.embed_dim = embed_dim
        self.n_head = n_head
        self.n_layer = n_layer
        self.ff_dim = ff_dim
        self.theta = theta

        self.embed = nn.Embedding(vocab_size, embed_dim)

        head_dim = embed_dim // n_head
        rope = RoPE(theta, head_dim, max_context_len)

        self.blocks = nn.ModuleList([
            Block(embed_dim, n_head, ff_dim, rope)
            for _ in range(n_layer)
        ])

        self.norm = nn.RMSNorm(embed_dim)
        self.unembed = nn.Linear(embed_dim, vocab_size, bias=False)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, ids, use_cache=False):
        x = self.embed(ids)
        for block in self.blocks:
            x = block(x, use_cache=use_cache)
        x = self.norm(x)
        logits = self.unembed(x)
        return logits

    def save(self, file_path):
        checkpoint = {
            'model_state_dict': self.state_dict(),
            'vocab_size': self.vocab_size,
            'max_context_len': self.max_context_len,
            'embed_dim': self.embed_dim,
            'n_head': self.n_head,
            'n_layer': self.n_layer,
            'ff_dim': self.ff_dim,
            'theta': self.theta,
        }
        torch.save(checkpoint, file_path)

    @classmethod
    def load_from(cls, file_path, device='cpu'):
        checkpoint = torch.load(file_path, map_location=device)

        model = cls(
            vocab_size=checkpoint['vocab_size'],
            max_context_len=checkpoint['max_context_len'],
            embed_dim=checkpoint['embed_dim'],
            n_head=checkpoint['n_head'],
            n_layer=checkpoint['n_layer'],
            ff_dim=checkpoint['ff_dim'],
            theta=checkpoint['theta']
        )

        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)

        return model

    def clear_cache(self):
        for block in self.blocks:
            block.clear_cache()

if __name__ == "__main__":
    vocab_size = 1000
    max_context_len = 256
    embed_dim = 384
    n_head = 6
    n_layer = 6
    ff_dim = int(embed_dim * 8 / 3)
    theta = 10000

    # モデルを作成
    model = GPT(vocab_size, max_context_len, embed_dim, n_head,
                n_layer, ff_dim, theta)
    # 動作テスト
    dummy_input = torch.randint(0, vocab_size, (1, max_context_len))
    logits = model(dummy_input)
    print(f"出力形状: {logits.shape}")


### `storybot/tokenizer.py`

SHA-256: `21f8b2d4c41c87df77f4118a8aa421250d198070f5ce0c03916237cb86498467`


In [ ]:
%%writefile storybot/tokenizer.py
import os
import pickle
from multiprocessing import Pool
import shutil
from collections import defaultdict
import regex as re
from tqdm import tqdm
import numpy as np


def pretokenize(text):
    pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    for m in re.finditer(pattern, text):
        yield m.group(0)

def count_pairs(ids, weight=1, counts=None):
    if counts is None:
        counts = defaultdict(int)

    for pair in zip(ids, ids[1:]):
        counts[pair] += weight
    return counts

def merge(ids, pair, new_id):
    merged_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            merged_ids.append(new_id)
            i += 2
        else:
            merged_ids.append(ids[i])
            i += 1
    return merged_ids

def find_chunk_boundaries(file_path, num_chunks, end_token="<|endoftext|>"):
    byte_end_token = end_token.encode("utf-8")

    with open(file_path, "rb") as file:  # ファイルをバイナリモードで開く
        # ファイルサイズを取得
        file.seek(0, os.SEEK_END)
        file_size = file.tell()
        file.seek(0)

        chunk_size = file_size // num_chunks

        # チャンクの開始位置を計算（等間隔）
        chunk_boundaries = [i * chunk_size for i in range(num_chunks)]
        chunk_boundaries.append(file_size)  # 最後にファイル終端を追加

        buffer_size = 4096  # 境界から先読みするバイト数

        # 境界位置の調整（終了トークンを探す）
        for bi in range(1, len(chunk_boundaries) - 1):
            chunk_position = chunk_boundaries[bi]
            file.seek(chunk_position)  # 境界の推定位置から開始

            while True:
                buffer = file.read(buffer_size)  # バッファサイズ分を読む

                # ファイル終端に達した場合
                if buffer == b"":
                    chunk_boundaries[bi] = file_size
                    break

                # 読み取ったチャンクで終了トークンを検索
                end_position = buffer.find(byte_end_token)
                if end_position != -1:
                    # 見つかった場合、その位置を新しい境界とする
                    chunk_boundaries[bi] = chunk_position + end_position
                    break

                # 見つからなかった場合、次のバッファ位置に移動
                chunk_position += buffer_size

    # 重複を除去し、ソートして返す
    return sorted(set(chunk_boundaries))

def process_single_chunk(file_path, start, end, end_token):
    """1つのチャンクを処理する関数"""
    pretoken_counts = defaultdict(int)

    # ファイルを開いてチャンクを読み込む
    with open(file_path, "rb") as f:
        f.seek(start)
        chunk_byte = f.read(end - start)
        chunk_text = chunk_byte.decode("utf-8", errors="ignore")

        # 特殊トークンで分割
        texts = chunk_text.split(end_token)

        # 各テキストを事前トークン化
        for text in texts:
            for pretoken in pretokenize(text):
                pretoken_counts[pretoken] += 1

    return pretoken_counts

def pretoken_chunk(args):
    file_path, start, end, end_token = args
    pretoken_counts = defaultdict(int)

    # ファイルを開いてチャンクを読み込む
    with open(file_path, "rb") as f:
        f.seek(start)
        chunk_byte = f.read(end - start)
        chunk_text = chunk_byte.decode("utf-8", errors="ignore")

        # 特殊トークンで分割
        texts = chunk_text.split(end_token)

        # 各テキストを事前トークン化
        for text in texts:
            for pretoken in pretokenize(text):
                pretoken_counts[pretoken] += 1

    return pretoken_counts

def train_bpe(file_path, vocab_size, end_token="<|endoftext|>", num_processes=8, num_chunks=8):
    # ステップ1: チャンクの準備
    chunk_boundaries = find_chunk_boundaries(file_path, num_chunks)
    total_chunks = len(chunk_boundaries) - 1

    chunk_info_list = []
    for i in range(total_chunks):
        start = chunk_boundaries[i]
        end = chunk_boundaries[i + 1]
        chunk_info_list.append((file_path, start, end, end_token))

    # ステップ2: 並列処理
    with Pool(processes=num_processes) as pool:
        all_results = list(tqdm(pool.imap(pretoken_chunk, chunk_info_list), total=len(chunk_info_list), desc="Pretokenizing"))

    # ステップ3: 結果を統合
    pretoken_counts = defaultdict(int)
    for chunk_result in all_results:
        for pretoken, count in chunk_result.items():
            pretoken_counts[pretoken] += count

    # 事前トークンをID列に変換
    ids_counts = {tuple(pretoken.encode("utf-8")): count for pretoken, count in pretoken_counts.items()}


    num_merges = vocab_size - 256 - 1
    merge_rules = {}
    pair_to_ids = defaultdict(set)  # キャッシュ

    pair_counts = defaultdict(int)
    for ids, count in ids_counts.items():
        count_pairs(ids, count, pair_counts)
        for pair in zip(ids, ids[1:]):  # キャッシュに登録
            pair_to_ids[pair].add(ids)

    for step in tqdm(range(num_merges), desc="Training BPE"):
        if not pair_counts:  # ペアが存在しない場合の処理
            break

        # 最頻出ペアを選択
        # best_pair = max(pair_counts, key=pair_counts.get)
        best_pair = max(pair_counts, key=lambda pair: (pair_counts[pair], pair[0], pair[1]))
        new_id = 256 + step
        merge_rules[best_pair] = new_id

        # best_pairを含むids列をキャッシュから取得
        affected_ids = pair_to_ids[best_pair]
        del pair_to_ids[best_pair]  # もう使わないので削除

        # 影響のあるID列だけを更新
        for ids in affected_ids:
            ids_count = ids_counts[tuple(ids)]
            new_ids = merge(ids, best_pair, new_id)

            del ids_counts[tuple(ids)]  # 古いID列を削除
            ids_counts[tuple(new_ids)] = ids_count  # 新しいID列を追加

            # 古いペア頻度を減少
            old_counts = count_pairs(ids)
            for pair, count in old_counts.items():
                pair_counts[pair] -= count * ids_count
                if pair_counts[pair] <= 0:
                    del pair_counts[pair]
                pair_to_ids[pair].discard(tuple(ids))

            # 新しいペア頻度を増加
            new_counts = count_pairs(new_ids)
            for pair, count in new_counts.items():
                pair_counts[pair] += count * ids_count
                pair_to_ids[pair].add(tuple(new_ids))

    return merge_rules

class BPETokenizer:
    def __init__(self, merge_rules, end_token="<|endoftext|>"):
        self.merge_rules = merge_rules
        self.end_token = end_token
        self.end_token_id = 256 + len(merge_rules)

        self.id_to_bytes = {i: bytes([i]) for i in range(256)}
        for (id1, id2), new_id in merge_rules.items():
            self.id_to_bytes[new_id] = self.id_to_bytes[id1] + self.id_to_bytes[id2]
        self.id_to_bytes[self.end_token_id] = self.end_token.encode("utf-8")

        self.vocab_size = len(self.id_to_bytes)

    @staticmethod
    def load_from(filepath):
        with open(filepath, "rb") as f:
            merge_rules = pickle.load(f)
        return BPETokenizer(merge_rules)

    def _encode_text(self, text):
        ids = list(text.encode("utf-8"))

        def get_merge_priority(pair):
            return self.merge_rules.get(pair, float('inf'))  # 存在しないペアは最低優先度

        while len(ids) > 1:
            # 現在のペアを取得（❶）
            counts = count_pairs(ids)

            # 最優先ペアを特定（❷）
            best_pair = min(counts, key=get_merge_priority)

            # マージ可能性の確認（❸）
            if best_pair not in self.merge_rules:
                break

            # マージの実行（❹）
            new_id = self.merge_rules[best_pair]
            ids = merge(ids, best_pair, new_id)

        return ids

    def encode(self, input_text, show_progress=False):
        pattern = '(' + re.escape(self.end_token) + ')'
        texts = re.split(pattern, input_text)
        all_ids = []

        # show_progressがTrueならtqdmで進捗表示
        texts = tqdm(texts) if show_progress else texts

        for text in texts:
            if text == self.end_token:
                all_ids.append(self.end_token_id)
            else:
                # 各事前トークンをBPEエンコード
                for pretoken in pretokenize(text):
                    ids = self._encode_text(pretoken)
                    all_ids.extend(ids)

        return all_ids

    def _encode_chunk(self, args):
        """チャンクを処理してディスクにキャッシュ"""
        file_path, start, end, cache_dir, chunk_idx = args

        with open(file_path, "rb") as f:
            f.seek(start)
            chunk_byte = f.read(end - start)
            chunk_text = chunk_byte.decode("utf-8", errors="ignore")

            # チャンクをエンコード
            ids = self.encode(chunk_text)

        # キャッシュファイルに保存
        cache_file = os.path.join(cache_dir, f"chunk_{chunk_idx:05d}.npy")
        np.array(ids, dtype=np.uint16).tofile(cache_file)

        return cache_file, len(ids)


    def encode_file(self, file_path, output_file,
                                    num_processes=4, num_chunks=64,
                                   cache_dir="bpe_cache"):

        # キャッシュディレクトリの準備
        os.makedirs(cache_dir, exist_ok=True)

        try:
            # ステップ1: チャンクを並列処理でトークナイズしてキャッシュ
            chunk_boundaries = find_chunk_boundaries(file_path, num_chunks)
            total_chunks = len(chunk_boundaries) - 1

            chunk_info_list = []
            for i in range(total_chunks):
                start = chunk_boundaries[i]
                end = chunk_boundaries[i + 1]
                chunk_info_list.append((file_path, start, end, cache_dir, i))

            with Pool(processes=num_processes) as pool:
                cache_results = list(tqdm(
                    pool.imap(self._encode_chunk, chunk_info_list),
                    total=len(chunk_info_list),
                    desc="Encoding chunks"
                ))

            # ステップ2: 総トークン数を計算
            cache_files = [r[0] for r in cache_results]
            token_counts = [r[1] for r in cache_results]
            total_tokens = sum(token_counts)

            # ステップ3: memmapファイルを作成
            dtype = np.uint16
            arr = np.memmap(output_file, dtype=dtype, mode='w+', shape=(total_tokens,))

            # ステップ4: バッチ処理でキャッシュからmemmapへ書き込み
            # OpenWebTextの例のようにバッチ化
            idx = 0
            for cache_file in cache_files:
                chunk_data = np.fromfile(cache_file, dtype=dtype)
                arr[idx : idx + len(chunk_data)] = chunk_data
                idx += len(chunk_data)

            arr.flush()
            del arr

        finally:
            # キャッシュは削除
            shutil.rmtree(cache_dir)

        return total_tokens

    def decode(self, ids):
        byte_list = [self.id_to_bytes[i] for i in ids]
        text_bytes = b"".join(byte_list)
        text = text_bytes.decode("utf-8", errors="replace")
        return text


### `storybot/utils.py`

SHA-256: `a02419af941e7646a89831931b7dcc33d97b67b60f041e0a9e4c5ea4ef70c500`


In [ ]:
%%writefile storybot/utils.py
import torch
import torch.nn.functional as F


@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=1000, temperature=1.0):
    model.eval()
    model.clear_cache()

    device = next(model.parameters()).device
    ids = tokenizer.encode(prompt)
    ids = torch.tensor([ids], dtype=torch.long, device=device)

    generated_ids = ids
    next_id = ids

    for _ in range(max_new_tokens):
        if ids.size(1) > model.max_context_len:
            ids = ids[:, -model.max_context_len:]

        logits = model(next_id, use_cache=True)[:, -1, :]  # kv cache
        if temperature == 0:
            next_id = logits.argmax(dim=-1, keepdim=True)
        else:
            probs = F.softmax(logits / temperature, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

        if next_id.item() == tokenizer.end_token_id:
            break

        ids = torch.cat((ids, next_id), dim=1)
        generated_ids = torch.cat((generated_ids, next_id), dim=1)

    # 終了トークンを除去
    generated_ids = generated_ids[generated_ids != tokenizer.end_token_id]

    generated_text = tokenizer.decode(generated_ids.tolist())
    return generated_text

def get_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    elif torch.backends.mps.is_available():
        return torch.device('mps')
    else:
        return torch.device('cpu')


## Complete chapter source


## `ch06/02_adamw.py`

SHA-256: `10df39a38e8774dd769e74dddcfd0fd28989d57dec8432c9679bb251fc9f3e73`


**Imports**


In [ ]:
import torch
from torch.optim.optimizer import Optimizer




**Definitions**


In [ ]:
class SGD(Optimizer):
    def __init__(self, params, lr=0.01):
        defaults = {'lr': lr}
        super().__init__(params, defaults)

    def step(self):
        for group in self.param_groups:
            lr = group['lr']

            for p in group['params']:
                if p.grad is None:
                    continue

                p.data = p.data - lr * p.grad.data


class AdamW(Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
        defaults = {"lr": lr,
                    "betas": betas,
                    "eps": eps,
                    "weight_decay": weight_decay}
        super().__init__(params, defaults)

    def step(self):
        for group in self.param_groups:
            beta1, beta2 = group['betas']

            for p in group['params']:
                if p.grad is None:
                    continue

                grad = p.grad.data
                state = self.state[p]

                if len(state) == 0:
                    state['t'] = 0
                    state['m'] = torch.zeros_like(p.data)
                    state['v'] = torch.zeros_like(p.data)

                state['t'] += 1
                t = state['t']

                # 1次モーメントと2次モーメントの更新
                m, v = state['m'], state['v']
                m = beta1 * m + (1 - beta1) * grad
                v = beta2 * v + (1 - beta2) * grad**2
                state['m'], state['v'] = m, v

                # バイアス補正
                m_hat = m / (1 - beta1**t)
                v_hat = v / (1 - beta2**t)

                lr, eps, wd = group['lr'], group['eps'], group['weight_decay']
                # パラメータ更新
                p.data = p.data - lr * m_hat / (v_hat.sqrt() + eps) - lr * wd * p.data




**Execution**


In [ ]:
torch.manual_seed(0)

# シンプルな線形モデル
model = torch.nn.Linear(2, 1)
# optimizer = AdamW(model.parameters(), lr=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.1)
# ダミーデータで学習
x = torch.tensor([[1.0, 2.0]])
y = torch.tensor([[3.0]])

# 数ステップ学習してlossの減少を確認
for step in range(5):
    output = model(x)
    loss = (output - y).pow(2).mean()

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    print(f"Step {step}: loss = {loss.item():.4f}")


## `ch06/03_lr_scheduler.py`

SHA-256: `11398f70299f1895738b029c9be15929c88d9b3c5719047c1b90d7f067d0cd60`


**Definitions**


In [ ]:
def get_lr(it, max_lr, warmup_iters, max_iters):
    # ウォームアップ：0 -> max_lr
    if it < warmup_iters:
        return max_lr * (it / warmup_iters)

    # アニーリング：max_lr -> 0
    if it < max_iters:
        progress = (it - warmup_iters) / (max_iters - warmup_iters)
        return max_lr * (1.0 - progress)

    return 0.0


## `ch06/04_mixed_precision.py`

SHA-256: `a87598f506d8a497a730c1cf8c6ac02b238c38d1d5807fc62398c4fb3c41dc81`


**Imports**


In [ ]:
import torch



**Execution**


In [ ]:
print("----- FP16 -----")

large = torch.tensor(1000.0, dtype=torch.float16)
small = torch.tensor(0.01, dtype=torch.float16)
print(large + small)  # tensor(1000., dtype=torch.float16)

tiny = torch.tensor(1e-8, dtype=torch.float16)
print(tiny)  # tensor(0., dtype=torch.float16)

huge = torch.tensor(70000.0, dtype=torch.float16)
print(huge)  # tensor(inf, dtype=torch.float16)

print("----- BF16 -----")

# FP16ではアンダーフロー
tiny_fp16 = torch.tensor(1e-8, dtype=torch.float16)
print(tiny_fp16)  # tensor(0., dtype=torch.float16)

# BF16では表現可能
tiny_bf16 = torch.tensor(1e-8, dtype=torch.bfloat16)
print(tiny_bf16)  # tensor(1.0012e-08, dtype=torch.bfloat16)

# FP16ではオーバーフロー
huge_fp16 = torch.tensor(70000.0, dtype=torch.float16)
print(huge_fp16)  # tensor(inf, dtype=torch.float16)

# BF16では表現可能
huge_bf16 = torch.tensor(70000.0, dtype=torch.bfloat16)
print(huge_bf16)  # tensor(70144., dtype=torch.bfloat16)


print("----- 自動混合精度 -----")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
a = torch.randn(1000, 1000, device=device)

with torch.autocast(device_type=device, dtype=torch.bfloat16):
    b = a @ a   # 行列積はBF16
    c = a.sum() # 累積はFP32
    print(b.dtype)  # torch.bfloat16
    print(c.dtype)  # torch.float32


## `ch06/05_pretrain.py`

SHA-256: `eab15c79765f1cf95811545265052e0dad43adffa884ff9fdbb2c525d7a5e8a9`


**Imports**


In [ ]:
import os, sys


**Execution**


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))
sys.path.append('.')



**Imports**


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.amp import autocast
from tqdm import tqdm
import matplotlib.pyplot as plt
from storybot.model import GPT
from storybot.tokenizer import BPETokenizer
from storybot.utils import get_device




**Definitions**


In [ ]:
def get_lr(it, max_lr, warmup_iters, max_iters):
    # ウォームアップ：0 -> max_lr
    if it < warmup_iters:
        return max_lr * (it / warmup_iters)

    # アニーリング：max_lr -> 0
    if it < max_iters:
        progress = (it - warmup_iters) / (max_iters - warmup_iters)
        return max_lr * (1.0 - progress)

    return 0.0


def get_batch(data, context_len, batch_size, device, random=True, offset=0):
    if random:
        ix = torch.randint(len(data) - context_len - 1, (batch_size,))
    else:
        ix = torch.arange(offset, offset + batch_size * context_len, context_len)

        ix = ix[ix + context_len + 1 < len(data)]
        if len(ix) == 0:
            return None, None

    # バッチを作成
    x = torch.stack([torch.from_numpy(data[i:i+context_len].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+context_len+1].astype(np.int64)) for i in ix])

    return x.to(device), y.to(device)

def evaluate(model, val_data, context_len, batch_size, device):
    """Validation: 全データを順番に処理"""
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    max_start = len(val_data) - context_len - 1
    num_batches = (max_start // context_len) // batch_size + 1

    with torch.no_grad():
        for batch_idx in tqdm(range(num_batches), desc="Validation"):
            offset = batch_idx * batch_size * context_len

            x, y = get_batch(val_data, context_len, batch_size, device,
                        random=False, offset=offset)

            if x is None:
                break

            with autocast(device_type=device.type, dtype=torch.bfloat16):
                logits = model(x)
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                    y.view(-1), reduction='sum')

            total_loss += loss.item()
            total_tokens += y.numel()

    model.train()
    return total_loss / total_tokens

# 設定


**Execution**


In [ ]:
device = get_device()
data_path = 'storybot/tiny_stories_train.bin'
val_data_path = 'storybot/tiny_stories_valid.bin'
tokenizer_path = 'storybot/merge_rules.pkl'
model_save_path = 'storybot/model_pretrain.pt'

# ハイパーパラメータ
context_len = 256
vocab_size = 10000
batch_size = 32
learning_rate = 0.001  # max_lr
warmup_iters = 200  # ウォームアップステップ数
max_iters = 40000
embed_dim = 512
n_head = 16
n_layer = 4
ff_dim = 1344
theta = 10000
eval_iters = 500
grad_clip = 1.0
save_iters = [500, 5000]  # 保存するイテレーションのリスト

# データをmemmapで読み込み
train_data = np.memmap(data_path, dtype=np.uint16, mode='r')
val_data = np.memmap(val_data_path, dtype=np.uint16, mode='r')

# トークナイザ、モデル、オプティマイザ
tokenizer = BPETokenizer.load_from(tokenizer_path)
model = GPT(
    vocab_size, context_len, embed_dim, n_head, n_layer, ff_dim, theta
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

total_params = sum(p.numel() for p in model.parameters())
print(f"パラメータ数: {total_params:,} ({total_params/1e6:.1f}M)")

pbar = tqdm(range(max_iters))

val_loss = float('inf')
val_losses = []
val_iters = []

for i in pbar:
    # 学習率を更新
    lr = get_lr(i, learning_rate, warmup_iters, max_iters)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    batch_x, batch_y = get_batch(train_data, context_len, batch_size, device)

    # 勾配をリセット
    optimizer.zero_grad()

    # 順伝播と損失計算(Mixed Precision)
    with autocast(device_type=device.type, dtype=torch.bfloat16):
        logits = model(batch_x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), batch_y.view(-1))

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()
    # 特定のイテレーションでモデルを保存
    if i in save_iters:
        save_path = f'storybot/model_iter_{i}.pt'
        model.save(save_path)
        print(f"\nモデルを保存しました（イテレーション {i}）: {save_path}")

    # 定期的に評価
    if (i % eval_iters) == 0 or i == max_iters - 1:
        val_loss = evaluate(model, val_data, context_len, batch_size, device)
        val_losses.append(val_loss)
        val_iters.append(i)
    pbar.set_postfix({'loss': f'{loss.item():.4f}', 'val_loss': f'{val_loss:.6f}'})


# Validation lossのグラフを描画
plt.figure(figsize=(10, 6))
plt.plot(val_iters, val_losses)
plt.xlabel('Iteration')
plt.ylabel('Validation Loss')
plt.grid(True)
plt.savefig('loss_val.png')

model.save(model_save_path)


## `ch06/06_generate.py`

SHA-256: `9d72e4f44cb25fdd944f2c2fef54e2f0334858b9e40fb9483048b6c5c7b80d02`


**Imports**


In [ ]:
import os
import sys


**Execution**


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))
sys.path.append('.')



**Imports**


In [ ]:
from storybot.model import GPT
from storybot.tokenizer import BPETokenizer
from storybot.utils import get_device, generate

# 設定


**Execution**


In [ ]:
device = get_device()
model_path = 'storybot/model_pretrain.pt'
tokenizer_path = 'storybot/merge_rules.pkl'

# 生成設定
# prompt = "Once upon a time"  # 生成の開始プロンプト
prompt = "<|endoftext|>"
max_new_tokens = 300  # 生成するトークン数の上限
temperature = 1.0  # 温度パラメータ（高いほどランダム）
num_samples = 3  # 生成するサンプル数

tokenizer = BPETokenizer.load_from(tokenizer_path)
model = GPT.load_from(model_path, device=device)

# テキスト生成
for i in range(num_samples):
    print(f"--- サンプル {i+1} ---")
    story = generate(
        model, tokenizer, prompt, max_new_tokens, temperature
    )
    print(story)


## `ch06/07_llm_judge.py`

SHA-256: `3d40e137de549e10c1a5cb1846ae634ea7320a8079fbbc659844ead85095e925`


**Imports**


In [ ]:
import os
import sys


**Execution**


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))
sys.path.append('.')



**Imports**


In [ ]:
import json
import statistics
from openai import OpenAI
from storybot.model import GPT
from storybot.tokenizer import BPETokenizer
from storybot.utils import get_device, generate


# 設定
# ==========================================


**Execution**


In [ ]:
client = OpenAI(api_key="your_api_key_here")
# ==========================================
device = get_device()
tokenizer_path = 'storybot/merge_rules.pkl'
tokenizer = BPETokenizer.load_from(tokenizer_path)

# 評価するモデルのパス（イテレーションごとに保存したもの）
model_paths = {
    500: 'storybot/model_iter_500.pt',
    5000: 'storybot/model_iter_5000.pt',
    40000: 'storybot/model_pretrain.pt',
}

# 生成設定
prompt = "<|endoftext|>"
max_new_tokens = 200
temperature = 1.0
num_samples = 10  # 各モデルで生成するサンプル数



**Definitions**


In [ ]:
def evaluate_story(client, story):
    """LLM-as-a-Judgeでストーリーを評価"""

    evaluation_prompt = f"""以下の子供向けストーリーを2つの観点で1-5点で評価してください。

ストーリー:
{story}

評価観点:
1. Coherence（一貫性）: 論理的につながっているか、物語として筋が通っているか
2. Grammar（文法）: 文法的に正しい英語か

以下のJSON形式で回答してください:
{{
    "coherence": <1-5の整数>,
    "grammar": <1-5の整数>,
    "comment": "<評価の簡単な理由>"
}}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": evaluation_prompt}],
        max_tokens=300,
        response_format={"type": "json_object"}
    )

    text = response.choices[0].message.content
    print("=====出力====")
    print(text)

    # response_formatを使えば、パース処理がシンプルになる
    return json.loads(text)



**Execution**


In [ ]:
results = {}
for iteration, model_path in model_paths.items():
    print(f"\n{'='*50}")
    print(f"Iteration {iteration}")
    print('='*50)

    model = GPT.load_from(model_path, device=device)
    iteration_results = []

    for i in range(num_samples):
        print(f"\n--- サンプル {i+1} ---")

        # ストーリー生成
        story = generate(model, tokenizer, prompt, max_new_tokens, temperature)
        print(f"Story: {story[:200]}...")

        # LLM-as-a-Judgeで評価
        scores = evaluate_story(client, story)
        print(f"Scores: {scores}")

        iteration_results.append({
            "story": story,
            "scores": scores
        })

    results[iteration] = iteration_results

# サマリー出力
print("\n" + "="*50)
print("Summary")
print("="*50)

for iteration in model_paths.keys():
    scores_list = [r["scores"] for r in results[iteration]]

    print(f"\nIteration {iteration}:")
    for key in ["coherence", "grammar"]:
        values = [s[key] for s in scores_list]
        avg = statistics.mean(values)
        std = statistics.stdev(values) if len(values) > 1 else 0
        print(f"  {key}: {avg:.2f} ± {std:.2f}")


## `ch06/08_dpo.py`

SHA-256: `4d39e88e2b254a47010a74843a3706cbf74e072c94f78a525fac244d0718ac8d`


**Imports**


In [ ]:
import os, sys


**Execution**


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))
sys.path.append('.')



**Imports**


In [ ]:
from itertools import cycle
import json
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from tqdm import tqdm
from storybot.model import GPT
from storybot.tokenizer import BPETokenizer
from storybot.utils import get_device

# 設定


**Execution**


In [ ]:
device = get_device()
data_path = 'storybot/tiny_stories_dpo.json'
tokenizer_path = 'storybot/merge_rules.pkl'
pretrain_model_path = 'storybot/model_pretrain.pt'
dpo_model_save_path = 'storybot/model_dpo.pt'

# ハイパーパラメータ
context_len = 256
batch_size = 8
learning_rate = 5e-6
beta = 0.1
max_iters = 1000




**Definitions**


In [ ]:
class DPODataset(Dataset):
    # コンストラクタ
    def __init__(self, data_path, tokenizer, context_len):
        self.tokenizer = tokenizer
        self.context_len = context_len
        self.samples = []

        with open(data_path) as f:
            data = json.load(f)

        for item in data:
            sample = self._create_sample(item['prompt'], item['chosen'], item['rejected'])
            self.samples.append(sample)

    # パディングとマスクの作成
    def _pad_and_mask(self, ids, prompt_len):
        mask = [0] * prompt_len + [1] * (len(ids) - prompt_len)

        if len(ids) > self.context_len:
            ids = ids[:self.context_len]
            mask = mask[:self.context_len]
        else:
            pad_len = self.context_len - len(ids)
            ids = ids + [0] * pad_len
            mask = mask + [0] * pad_len

        return ids, mask

    # サンプル作成
    def _create_sample(self, prompt, chosen, rejected):
        prompt_ids = self.tokenizer.encode(prompt)
        chosen_ids = prompt_ids + self.tokenizer.encode(chosen)
        rejected_ids = prompt_ids + self.tokenizer.encode(rejected)

        prompt_len = len(prompt_ids)
        chosen_ids, chosen_mask = self._pad_and_mask(chosen_ids, prompt_len)
        rejected_ids, rejected_mask = self._pad_and_mask(rejected_ids, prompt_len)

        return chosen_ids, chosen_mask, rejected_ids, rejected_mask

    # DataLoader用メソッド
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        chosen_ids, chosen_mask, rejected_ids, rejected_mask = self.samples[idx]
        return (
            torch.tensor(chosen_ids, dtype=torch.long),
            torch.tensor(chosen_mask, dtype=torch.long),
            torch.tensor(rejected_ids, dtype=torch.long),
            torch.tensor(rejected_mask, dtype=torch.long),
        )


def get_sequence_logprobs(model, ids, mask):
    logits = model(ids)  # (B, C, V)
    log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)  # (B, C-1, V)
    labels = ids[:, 1:]  # (B, C-1)

    per_token_logprobs = torch.gather(
        log_probs, dim=-1, index=labels.unsqueeze(-1)
    ).squeeze(-1)  # (B, C-1)
    # マスクを適用（応答部分のみ）
    masked_logprobs = per_token_logprobs * mask[:, 1:]
    return masked_logprobs.sum(dim=-1)  # (B,)


def compute_dpo_loss(model, ref_model, chosen_ids, chosen_mask, rejected_ids, rejected_mask, beta):
    # 現在のモデルのlog-prob
    chosen_logprobs = get_sequence_logprobs(model, chosen_ids, chosen_mask)
    rejected_logprobs = get_sequence_logprobs(model, rejected_ids, rejected_mask)

    # 参照モデルのlog-prob
    with torch.no_grad():
        ref_chosen_logprobs = get_sequence_logprobs(ref_model, chosen_ids, chosen_mask)
        ref_rejected_logprobs = get_sequence_logprobs(ref_model, rejected_ids, rejected_mask)

    # DPO loss
    logits = beta * (
        (chosen_logprobs - rejected_logprobs) -
        (ref_chosen_logprobs - ref_rejected_logprobs)
    )
    return -F.logsigmoid(logits).mean()


# トークナイザとデータセットの準備


**Execution**


In [ ]:
tokenizer = BPETokenizer.load_from(tokenizer_path)
dataset = DPODataset(data_path, tokenizer, context_len)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# モデルとオプティマイザ
model = GPT.load_from(pretrain_model_path, device=device)
ref_model = GPT.load_from(pretrain_model_path, device=device)
ref_model.eval()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# 学習ループ
losses = []
data_iter = cycle(dataloader)
pbar = tqdm(range(max_iters))

for i in pbar:
    chosen_ids, chosen_mask, rejected_ids, rejected_mask = next(data_iter)
    chosen_ids, chosen_mask = chosen_ids.to(device), chosen_mask.to(device)
    rejected_ids, rejected_mask = rejected_ids.to(device), rejected_mask.to(device)

    loss = compute_dpo_loss(
        model, ref_model,
        chosen_ids, chosen_mask,
        rejected_ids, rejected_mask,
        beta
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    pbar.set_postfix({'loss': f'{loss.item():.4f}'})

# 結果を保存
plt.figure(figsize=(10, 6))
plt.plot(losses)
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.grid(True)
plt.savefig("loss_dpo.png", bbox_inches='tight')

model.save(dpo_model_save_path)


## `ch06/09_llm_judge.py`

SHA-256: `028eeedfadbaad116b6d33255b0c7dc908470d66cf204ec0e955750c3d0818b9`


**Imports**


In [ ]:
import os
import sys


**Execution**


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))
sys.path.append('.')



**Imports**


In [ ]:
import json
import torch
import torch.nn.functional as F
from openai import OpenAI
from storybot.model import GPT
from storybot.tokenizer import BPETokenizer
from storybot.utils import get_device, generate

# 設定
# ==========================================


**Execution**


In [ ]:
client = OpenAI(api_key="your_api_key_here")
# ==========================================
device = get_device()
tokenizer_path = 'storybot/merge_rules.pkl'
tokenizer = BPETokenizer.load_from(tokenizer_path)

# 比較するモデル
model_paths = {
    'pretrain': 'storybot/model_pretrain.pt',
    'dpo': 'storybot/model_dpo.pt',
}

# 評価設定
prompt = "Once upon a time"
num_comparisons = 100  # 比較回数
max_new_tokens = 150
temperature = 1.0




**Definitions**


In [ ]:
def compare_stories(client, story_a, story_b):
    """2つのストーリーを比較し、どちらがよりハッピーエンドかを判定"""

    evaluation_prompt = f"""以下の2つの子供向けストーリーを比較し、どちらがよりハッピーエンドかを判定してください。

【Story A】
{story_a}

【Story B】
{story_b}

どちらがより明るく幸せな結末か、または希望に満ちた内容かを判断してください。
JSON形式で回答: {{"winner": "A" or "B" or "tie", "reason": "簡潔な理由"}}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": evaluation_prompt}],
        max_tokens=150,
        response_format={"type": "json_object"}
    )

    text = response.choices[0].message.content
    return json.loads(text)

# モデルをロード


**Execution**


In [ ]:
model_pretrain = GPT.load_from(model_paths['pretrain'], device=device)
model_dpo = GPT.load_from(model_paths['dpo'], device=device)

# 結果を記録
results = []
wins = {"pretrain": 0, "dpo": 0, "tie": 0}

for i in range(num_comparisons):
    print(f"\n{'='*60}")
    print(f"Comparison {i+1}/{num_comparisons}")
    print('='*60)

    # 両モデルでストーリーを生成
    story_pretrain = generate(model_pretrain, tokenizer, prompt, max_new_tokens, temperature)
    story_dpo = generate(model_dpo, tokenizer, prompt, max_new_tokens, temperature)

    print(f"\n[Pretrain]: {story_pretrain[:100]}...")
    print(f"\n[DPO]: {story_dpo[:100]}...")

    # 位置バイアスを避けるため、ランダムに順序を入れ替え
    import random
    if random.random() < 0.5:
        story_a, story_b = story_pretrain, story_dpo
        mapping = {"A": "pretrain", "B": "dpo"}
    else:
        story_a, story_b = story_dpo, story_pretrain
        mapping = {"A": "dpo", "B": "pretrain"}

    # LLM-as-a-Judgeで比較
    judgment = compare_stories(client, story_a, story_b)

    winner_label = judgment["winner"]
    if winner_label == "tie":
        winner = "tie"
    else:
        winner = mapping[winner_label]

    wins[winner] += 1

    print(f"\n🏆 Winner: {winner}")
    print(f"   Reason: {judgment['reason']}")

    results.append({
        "story_pretrain": story_pretrain,
        "story_dpo": story_dpo,
        "winner": winner,
        "reason": judgment["reason"]
    })

# サマリー出力
print("\n" + "="*60)
print("📊 PAIRWISE COMPARISON RESULTS")
print("="*60)

total = num_comparisons
print(f"\n  Pretrain wins: {wins['pretrain']:3d} ({wins['pretrain']/total*100:5.1f}%)")
print(f"  DPO wins:      {wins['dpo']:3d} ({wins['dpo']/total*100:5.1f}%)")
print(f"  Ties:          {wins['tie']:3d} ({wins['tie']/total*100:5.1f}%)")

# 勝率（tieを除く）
if wins['pretrain'] + wins['dpo'] > 0:
    dpo_winrate = wins['dpo'] / (wins['pretrain'] + wins['dpo']) * 100
    print(f"\n  DPO win rate (excluding ties): {dpo_winrate:.1f}%")


## `ch06/lr_graph.py`

SHA-256: `ee96f127a61ffffa9294ee683bc3cde384df57c87d26ab61bf846f08ee754cf2`


**Imports**


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 日本語フォント設定


**Execution**


In [ ]:
plt.rcParams['font.family'] = 'Hiragino Sans'  # macOS
# plt.rcParams['font.family'] = 'Yu Gothic'  # Windows

# パラメータ
warmup_ratio = 0.05  # ウォームアップ期間（全体の5%）
eta_min_ratio = 0.1  # コサインアニーリングの最小学習率（最大の10%）

# データ生成
t = np.linspace(0, 1, 1000)

# コサインアニーリング（ウォームアップ付き）


**Definitions**


In [ ]:
def cosine_annealing(t, warmup_ratio, eta_min_ratio):
    lr = np.zeros_like(t)
    for i, ti in enumerate(t):
        if ti < warmup_ratio:
            # ウォームアップ: 0 -> 1
            lr[i] = ti / warmup_ratio
        else:
            # コサインアニーリング: 1 -> eta_min
            progress = (ti - warmup_ratio) / (1 - warmup_ratio)
            lr[i] = eta_min_ratio + 0.5 * (1 - eta_min_ratio) * (1 + np.cos(np.pi * progress))
    return lr

# D2Z（ウォームアップ付き）
def d2z(t, warmup_ratio):
    lr = np.zeros_like(t)
    for i, ti in enumerate(t):
        if ti < warmup_ratio:
            # ウォームアップ: 0 -> 1
            lr[i] = ti / warmup_ratio
        else:
            # 線形減衰: 1 -> 0
            progress = (ti - warmup_ratio) / (1 - warmup_ratio)
            lr[i] = 1 - progress
    return lr



**Execution**


In [ ]:
cosine_lr = cosine_annealing(t, warmup_ratio, eta_min_ratio)
d2z_lr = d2z(t, warmup_ratio)

# プロット作成
fig, ax = plt.subplots(figsize=(10, 6))

# ウォームアップ領域をグレーで塗りつぶし
ax.axvspan(0, warmup_ratio, color='lightgray', alpha=0.5)
ax.text(0.01, 1.02, 'ウォームアップ', fontsize=10, va='bottom')

# 学習率曲線
ax.plot(t, cosine_lr, 'b-', linewidth=2, label='コサインアニーリング')
ax.plot(t, d2z_lr, color='orange', linestyle='--', linewidth=2, label='D2Z')

# 軸の設定
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.set_xlabel('学習の進行度', fontsize=12)
ax.set_ylabel('学習率', fontsize=12)

# グリッド
ax.grid(True, linestyle='--', alpha=0.7)

# 凡例
ax.legend(loc='upper right', fontsize=12)

# 余白調整
plt.tight_layout()

# PNGで保存
plt.savefig('lr_schedule.png', format='png', bbox_inches='tight')
plt.close()


## T4 execution note

The implementation above keeps the upstream code and hyperparameters intact. For chapters with long training loops, a Colab T4 can execute the implementation, but completing the full training schedule may take substantial wall-clock time. No reduced model, shortened algorithm, or toy substitute is enabled by default in this notebook.
